# Chapter 3 — Matching from the Dispersion Suppressor to the Straight Section

In Chapter 2 we built a forward dispersion suppressor, `DISPSUPF`, so that the horizontal dispersion and its derivative vanish before the straight section.

In this chapter we add a Twiss matching section after `DISPSUPF`. The goal is to make the Twiss parameters at the end of the matching section equal to the periodic Twiss parameters of the straight-section FODO cell:

$$
(\beta_a, \alpha_a, \beta_b, \alpha_b)_\mathrm{end}
=
(\beta_a, \alpha_a, \beta_b, \alpha_b)_\mathrm{SS, periodic}.
$$

We do this by adding four special quadrupoles:

```text
QFF2, QDF2, QFF3, QDF3
```

Their strengths are the four optimization variables. Conceptually the line is

```text
arc → DISPSUPF → MSSF → straight-section FODO
```

Here `MSSF` is the new matching section, and `ARC_TO_SSF = (DISPSUPF, MSSF)`.

## 1. Load packages

The core calculation below uses only transfer matrices, so it is runnable in a plain Julia notebook. If `SciBmad` is installed, the last section also builds the same matching section as SciBmad elements.

In [ ]:
using LinearAlgebra
using Printf

HAS_SCIBMAD = try
    @eval using SciBmad
    true
catch err
    @warn "SciBmad was not loaded. The transfer-matrix matching cells below still run."
    false
end

HAS_PLOTS = try
    @eval using Plots
    true
catch err
    @warn "Plots.jl was not loaded. Plotting cells will be skipped."
    false
end

## 2. Linear optics tools

For this chapter, the essential operation is Twiss propagation. We use uncoupled 2-by-2 transfer matrices in the horizontal and vertical planes.

For a Twiss pair $(\beta, \alpha)$, define

$$
\gamma = \frac{1 + \alpha^2}{\beta}.
$$

Then the Twiss matrix is

$$
\Sigma = \begin{pmatrix}
\beta & -\alpha \\
-\alpha & \gamma
\end{pmatrix}.
$$

After a transfer matrix $M$, the propagated Twiss matrix is

$$
\Sigma_2 = M \Sigma_1 M^T.
$$

In [ ]:
struct Twiss
    beta::Float64
    alpha::Float64
end

gamma(t::Twiss) = (1 + t.alpha^2) / t.beta

function twiss_matrix(t::Twiss)
    return [t.beta -t.alpha; -t.alpha gamma(t)]
end

function propagate_twiss(t::Twiss, M::AbstractMatrix)
    S = M * twiss_matrix(t) * transpose(M)
    return Twiss(S[1, 1], -S[1, 2])
end

function drift_matrix(L)
    return [1.0 L; 0.0 1.0]
end

function quad_matrix(k, L)
    if abs(k) < 1e-14
        return drift_matrix(L)
    elseif k > 0
        r = sqrt(k)
        return [cos(r * L) sin(r * L) / r; -r * sin(r * L) cos(r * L)]
    else
        r = sqrt(-k)
        return [cosh(r * L) sinh(r * L) / r; r * sinh(r * L) cosh(r * L)]
    end
end

struct Element
    name::String
    kind::Symbol
    L::Float64
    K1::Float64
end

drift(name, L) = Element(name, :drift, L, 0.0)
bend(name, L) = Element(name, :bend, L, 0.0)
quad(name, L, K1) = Element(name, :quad, L, K1)

function element_matrix(e::Element, plane::Symbol)
    if e.kind == :quad
        # In an uncoupled quadrupole, the vertical focusing strength has the opposite sign.
        k = plane == :x ? e.K1 : -e.K1
        return quad_matrix(k, e.L)
    else
        # For this matching notebook we keep the bend length in the transverse transport.
        # The bend's role in Chapter 2 was dispersion suppression; here the target is Twiss matching.
        return drift_matrix(e.L)
    end
end

function line_matrix(line, plane::Symbol)
    M = Matrix(I, 2, 2)
    for e in line
        M = element_matrix(e, plane) * M
    end
    return M
end

function phase_advance(M)
    c = clamp(tr(M) / 2, -1.0, 1.0)
    return acos(c)
end

function periodic_twiss(line, plane::Symbol)
    M = line_matrix(line, plane)
    μ = phase_advance(M)
    sμ = sin(μ)
    if abs(sμ) < 1e-12
        error("Phase advance is too close to 0 or π; periodic Twiss is ill-conditioned.")
    end
    β = M[1, 2] / sμ
    α = (M[1, 1] - M[2, 2]) / (2 * sμ)
    return Twiss(β, α), μ
end

## 3. Define the FODO geometry

We use the same geometric parameters as before:

```text
Q length: 0.5 m
D1:       0.609 m
D2:       1.241 m
B chord:  6.86 m
DB:       5.855 m
```

`DB` is the straight-section drift that replaces the arc bend space.

In [ ]:
LQ = 0.5
D1 = 0.609
D2 = 1.241
LB = 6.86
DB = 5.855

function fodo_like(qf_name, kqf, qd_name, kqd, bend_name, bend_length; forward=true)
    if forward
        return Element[
            quad(qf_name, LQ, kqf),
            drift("D1", D1),
            bend(bend_name, bend_length),
            drift("D2", D2),
            quad(qd_name, LQ, kqd),
            drift("D1", D1),
            bend(bend_name, bend_length),
            drift("D2", D2),
        ]
    else
        return Element[
            quad(qf_name, LQ, kqf),
            drift("D2", D2),
            bend(bend_name, bend_length),
            drift("D1", D1),
            quad(qd_name, LQ, kqd),
            drift("D2", D2),
            bend(bend_name, bend_length),
            drift("D1", D1),
        ]
    end
end

function find_k_for_90deg(make_line; lo=0.01, hi=1.0, n_iter=100)
    target = π / 2
    for _ in 1:n_iter
        mid = (lo + hi) / 2
        _, μ = periodic_twiss(make_line(mid), :x)
        if μ < target
            lo = mid
        else
            hi = mid
        end
    end
    return (lo + hi) / 2
end

K_arc = find_k_for_90deg(k -> fodo_like("QF", k, "QD", -k, "B", LB; forward=true))
K_ss  = find_k_for_90deg(k -> fodo_like("QFSS", k, "QDSS", -k, "DB", DB; forward=true))

@printf("Arc FODO 90-degree K1  ≈ %.15f\n", K_arc)
@printf("SS FODO 90-degree K1   ≈ %.15f\n", K_ss)

The straight-section FODO cell gives the target Twiss parameters for this chapter.

In [ ]:
FODOSSF = fodo_like("QFSS", K_ss, "QDSS", -K_ss, "DB", DB; forward=true)

target_a, μa_ss = periodic_twiss(FODOSSF, :x)
target_b, μb_ss = periodic_twiss(FODOSSF, :y)

println("Straight-section periodic Twiss target:")
@printf("  beta.a  = %.10f, alpha.a = %.10f, phase.a = %.10f rad\n", target_a.beta, target_a.alpha, μa_ss)
@printf("  beta.b  = %.10f, alpha.b = %.10f, phase.b = %.10f rad\n", target_b.beta, target_b.alpha, μb_ss)

These values are the matching target for the end of `ARC_TO_SSF`.

So the residual vector will be

$$
r = \begin{pmatrix}
(\beta_a - \beta_{a,\mathrm{target}})/\beta_{a,\mathrm{target}} \\
\alpha_a - \alpha_{a,\mathrm{target}} \\
(\beta_b - \beta_{b,\mathrm{target}})/\beta_{b,\mathrm{target}} \\
\alpha_b - \alpha_{b,\mathrm{target}}
\end{pmatrix}.
$$

The beta residuals are scaled relatively so that beta and alpha terms have comparable sizes.

## 4. Build the Chapter 2 dispersion suppressor

Chapter 2 created a suppressor from two half-bend FODO-like cells. The first cell uses the ordinary arc quadrupoles. The second cell uses the special quadrupoles `QFF1` and `QDF1`, which were optimized to make the dispersion vanish.

For the Twiss matching step, we take the Chapter 2 optimized values as the incoming line to Chapter 3.

In [ ]:
# Values from the Chapter 2 matching result. Replace these with your own optimized values if different.
K_QFF1 = 0.31279
K_QDF1 = -0.31244

FODOAF = fodo_like("QF", K_arc, "QD", -K_arc, "B", LB; forward=true)

twiss_arc_a, _ = periodic_twiss(FODOAF, :x)
twiss_arc_b, _ = periodic_twiss(FODOAF, :y)

DISPSUPF = vcat(
    fodo_like("QF", K_arc, "QD", -K_arc, "BH", LB; forward=true),
    fodo_like("QFF1", K_QFF1, "QDF1", K_QDF1, "BH", LB; forward=true),
)

twiss_after_dispsup_a = propagate_twiss(twiss_arc_a, line_matrix(DISPSUPF, :x))
twiss_after_dispsup_b = propagate_twiss(twiss_arc_b, line_matrix(DISPSUPF, :y))

println("Twiss at the end of DISPSUPF, before Chapter 3 matching:")
@printf("  beta.a  = %.10f, alpha.a = %.10f\n", twiss_after_dispsup_a.beta, twiss_after_dispsup_a.alpha)
@printf("  beta.b  = %.10f, alpha.b = %.10f\n", twiss_after_dispsup_b.beta, twiss_after_dispsup_b.alpha)

## 5. Add the Chapter 3 matching section

The forward matching section is two straight-section-like cells. The four quadrupoles are special matching quadrupoles:

```text
MSSF = QFF2 ... QDF2 ... QFF3 ... QDF3 ...
```

The variables are their $K_1$ strengths:

$$
x = (K_{QFF2}, K_{QDF2}, K_{QFF3}, K_{QDF3}).
$$

In [ ]:
function MSSF(k; forward=true)
    return vcat(
        fodo_like("QFF2", k[1], "QDF2", k[2], "DB", DB; forward=forward),
        fodo_like("QFF3", k[3], "QDF3", k[4], "DB", DB; forward=forward),
    )
end

function end_twiss_after_MSSF(k)
    line = MSSF(k; forward=true)
    end_a = propagate_twiss(twiss_after_dispsup_a, line_matrix(line, :x))
    end_b = propagate_twiss(twiss_after_dispsup_b, line_matrix(line, :y))
    return end_a, end_b
end

function residual(k)
    end_a, end_b = end_twiss_after_MSSF(k)
    return [
        (end_a.beta - target_a.beta) / target_a.beta,
        end_a.alpha - target_a.alpha,
        (end_b.beta - target_b.beta) / target_b.beta,
        end_b.alpha - target_b.alpha,
    ]
end

K_start = [K_ss, -K_ss, K_ss, -K_ss]
println("Initial residual using ordinary straight-section strengths:")
println(residual(K_start))

## 6. Optimize the four quadrupoles

We use a small Levenberg-Marquardt style least-squares optimizer. This mirrors the structure of the accelerator matching problem: use four knobs to make four residuals vanish.

In [ ]:
function finite_jacobian(f, x; h=1e-6)
    r0 = f(x)
    J = zeros(length(r0), length(x))
    for j in eachindex(x)
        dx = zeros(length(x))
        dx[j] = h * max(1.0, abs(x[j]))
        J[:, j] = (f(x + dx) - f(x - dx)) / (2 * dx[j])
    end
    return J
end

function lm_optimize(f, x0; weights=ones(length(f(x0))), maxiter=100, λ0=1e-3, tol=1e-12)
    x = copy(x0)
    λ = λ0
    history = NamedTuple[]
    wf(z) = sqrt.(weights) .* f(z)

    for iter in 1:maxiter
        r = wf(x)
        merit = sum(abs2, r)
        J = finite_jacobian(wf, x)
        A = transpose(J) * J + λ * Matrix(I, length(x), length(x))
        step = -A \ (transpose(J) * r)
        x_trial = x + step
        merit_trial = sum(abs2, wf(x_trial))

        if merit_trial < merit
            x = x_trial
            λ = max(λ / 3, 1e-20)
        else
            λ = min(λ * 10, 1e20)
        end

        push!(history, (iter=iter, merit=merit_trial, lambda=λ, step_norm=norm(step)))

        if norm(step) < tol || merit_trial < tol^2
            break
        end
    end

    return x, history
end

K_match, hist = lm_optimize(residual, K_start; maxiter=100)

println("Optimized matching quadrupoles:")
@printf("  QFF2 K1 = %.15f\n", K_match[1])
@printf("  QDF2 K1 = %.15f\n", K_match[2])
@printf("  QFF3 K1 = %.15f\n", K_match[3])
@printf("  QDF3 K1 = %.15f\n", K_match[4])
println("\nFinal residual:")
println(residual(K_match))
println("\nFinal merit = ", sum(abs2, residual(K_match)))

## 7. Check the matched Twiss parameters

The end of `ARC_TO_SSF` should now match the straight-section periodic Twiss.

In [ ]:
matched_a, matched_b = end_twiss_after_MSSF(K_match)

println("Matched Twiss at END compared with straight-section target:")
println()
@printf("%-10s %18s %18s %18s\n", "quantity", "target", "matched", "difference")
@printf("%-10s %18.10e %18.10e %18.10e\n", "beta.a",  target_a.beta,  matched_a.beta,  matched_a.beta  - target_a.beta)
@printf("%-10s %18.10e %18.10e %18.10e\n", "alpha.a", target_a.alpha, matched_a.alpha, matched_a.alpha - target_a.alpha)
@printf("%-10s %18.10e %18.10e %18.10e\n", "beta.b",  target_b.beta,  matched_b.beta,  matched_b.beta  - target_b.beta)
@printf("%-10s %18.10e %18.10e %18.10e\n", "alpha.b", target_b.alpha, matched_b.alpha, matched_b.alpha - target_b.alpha)

## 8. Plot the beta functions through the line

This plot shows the beta functions through the dispersion suppressor plus the matching section.

In [ ]:
function propagate_line(t0::Twiss, line, plane::Symbol)
    s = Float64[0.0]
    beta = Float64[t0.beta]
    alpha = Float64[t0.alpha]
    names = String["START"]
    t = t0

    for e in line
        t = propagate_twiss(t, element_matrix(e, plane))
        push!(s, s[end] + e.L)
        push!(beta, t.beta)
        push!(alpha, t.alpha)
        push!(names, e.name)
    end
    return (s=s, beta=beta, alpha=alpha, names=names)
end

ARC_TO_SSF = vcat(DISPSUPF, MSSF(K_match; forward=true))
track_a = propagate_line(twiss_arc_a, ARC_TO_SSF, :x)
track_b = propagate_line(twiss_arc_b, ARC_TO_SSF, :y)

if HAS_PLOTS
    plot(track_a.s, track_a.beta, label="βa", xlabel="s [m]", ylabel="β [m]", title="Beta functions through ARC_TO_SSF")
    plot!(track_b.s, track_b.beta, label="βb")
end

## 9. Optional: build the same matching section with SciBmad elements

The optimization above is intentionally transparent: the merit function and the Twiss propagation are visible. Once the matching strengths are known, we can build the same `MSSF` line with SciBmad-style elements.

In [ ]:
if HAS_SCIBMAD
    QFF2_ele = Quadrupole(L=LQ, Kn1=K_match[1])
    QDF2_ele = Quadrupole(L=LQ, Kn1=K_match[2])
    QFF3_ele = Quadrupole(L=LQ, Kn1=K_match[3])
    QDF3_ele = Quadrupole(L=LQ, Kn1=K_match[4])
    D1_ele = Drift(L=D1)
    D2_ele = Drift(L=D2)
    DB_ele = Drift(L=DB)

    mssf_sci = Beamline([
        QFF2_ele, D1_ele, DB_ele, D2_ele,
        QDF2_ele, D1_ele, DB_ele, D2_ele,
        QFF3_ele, D1_ele, DB_ele, D2_ele,
        QDF3_ele, D1_ele, DB_ele, D2_ele,
    ], species_ref=Species("electron"), E_ref=18e9)

    mssf_sci
else
    println("SciBmad is not loaded, so this optional section was skipped.")
end

## 10. Save the matching strengths

This writes a small Bmad-style snippet and a Julia snippet. You can paste these values into the next chapter's lattice construction.

In [ ]:
open("mSSF_quads.bmad", "w") do io
    @printf(io, "QFF2: Quadrupole, L = %.6f, K1 = %.15g\n", LQ, K_match[1])
    @printf(io, "QDF2: Quadrupole, L = %.6f, K1 = %.15g\n", LQ, K_match[2])
    @printf(io, "QFF3: Quadrupole, L = %.6f, K1 = %.15g\n", LQ, K_match[3])
    @printf(io, "QDF3: Quadrupole, L = %.6f, K1 = %.15g\n", LQ, K_match[4])
end

open("mSSF_values.jl", "w") do io
    @printf(io, "K_QFF2 = %.17g\n", K_match[1])
    @printf(io, "K_QDF2 = %.17g\n", K_match[2])
    @printf(io, "K_QFF3 = %.17g\n", K_match[3])
    @printf(io, "K_QDF3 = %.17g\n", K_match[4])
end

println("Wrote mSSF_quads.bmad and mSSF_values.jl")

## Exercises

### Exercise 1 — Reversed matching section

Construct the reverse matching section by following the same logic as the forward section. Use the reverse ordering of drifts inside each FODO-like cell:

```text
QF, D2, DB, D1, QD, D2, DB, D1
```

Then optimize the reverse matching quadrupoles and compare their values with the forward matching values.

Use the name `mSSR` or `MSSR` for the reverse matching section.

### Exercise 2 — Merit weight

Restart from the unoptimized matching section and make the first beta residual much more important than the others. In the code above, this corresponds to changing the weights to something like

```julia
weights = [1e10, 1, 1, 1]
```

Then compare the convergence history and final residuals with the equal-weight optimization.

In [ ]:
# Exercise 2 starter cell
# Try running this and compare with the equal-weight result above.

bad_weights = [1e10, 1.0, 1.0, 1.0]
K_bad, hist_bad = lm_optimize(residual, K_start; weights=bad_weights, maxiter=100)

println("Result with beta.a weight = 1e10:")
@printf("  QFF2 K1 = %.15f\n", K_bad[1])
@printf("  QDF2 K1 = %.15f\n", K_bad[2])
@printf("  QFF3 K1 = %.15f\n", K_bad[3])
@printf("  QDF3 K1 = %.15f\n", K_bad[4])
println("\nUnweighted residual at the final point:")
println(residual(K_bad))
println("\nWeighted merit at the final point = ", sum(abs2, sqrt.(bad_weights) .* residual(K_bad)))

## Summary

The Chapter 3 construction is:

```text
DISPSUPF + MSSF = ARC_TO_SSF
```

`DISPSUPF` comes from Chapter 2. `MSSF` is new in this chapter and contains four special quadrupoles:

```text
QFF2, QDF2, QFF3, QDF3
```

These four quadrupole strengths are optimized so that at the end of the line we have

$$
\beta_a = \beta_{a,\mathrm{SS}},\quad
\alpha_a = \alpha_{a,\mathrm{SS}},\quad
\beta_b = \beta_{b,\mathrm{SS}},\quad
\alpha_b = \alpha_{b,\mathrm{SS}}.
$$

After this matching section, the particle can enter the straight-section FODO cells without a Twiss mismatch.